# Can we predict whether a student passes the closing quiz?
Target: `passed_quiz`. Follows `checklist.md`.

Design decisions (made before any modelling):
- `student_key` repeats across rows (about 7 sessions per student), so every split keeps whole students together (checklist 1, 2).
- There is no session date column (`enrol_year` is the student's enrolment year, not a session time), so a time-ordered split is not possible; a grouped split is the right structure.
- `student_key` is an identifier and is dropped from features. `school_code` is a code (not a quantity) and is left out of the candidate features.
- `session_minutes` and `hints_used` describe the session in progress; I treat them as available when the prediction is made (just before the quiz that closes the session). If the intended use is to predict *before* the session starts, they would not be known, so I also report a feature set without them.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from sklearn.base import clone
df=pd.read_csv('data.csv'); print(df.shape, df.passed_quiz.mean())
print(df.groupby('student_key').prior_score.nunique().describe())

`prior_score` is a placement test score yet varies between a student's rows ("as stored on the session record"), so it may be re-recorded per session. Its correlation with the target is near zero, so it carries little risk or value either way; it is kept as a candidate feature but noted as a caveat.

## 1. Test set: hold out whole students (20%)

In [ ]:
gss=GroupShuffleSplit(n_splits=1,test_size=0.2,random_state=0)
tr,te=next(gss.split(df,groups=df.student_key))
train,test=df.iloc[tr].reset_index(drop=True),df.iloc[te].reset_index(drop=True)
assert not set(train.student_key)&set(test.student_key)
print(len(train),len(test),train.passed_quiz.mean())

## 2. Candidates, named before scoring
Models: `baseline` (predict the training prior), `logreg` (scaled logistic regression), `hgb` (gradient boosting, small grid: `max_depth` in {2,3}, `learning_rate` in {0.05,0.1}).
Feature sets: `with_session` = prior_score, enrol_year, session_minutes, hints_used; `pre_session` = prior_score, enrol_year.
Selection metric: CV ROC AUC (threshold-free). The decision threshold is then chosen on out-of-fold predictions to maximise accuracy.
Cross-validation: 5-fold `GroupKFold` on students within the training rows; preprocessing (scaling) is inside the pipeline so it is fit per fold.

In [ ]:
FS={'with_session':['prior_score','enrol_year','session_minutes','hints_used'],
    'pre_session':['prior_score','enrol_year']}
MODELS={'baseline':DummyClassifier(strategy='prior'),
 'logreg':make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000))}
for d in (2,3):
    for lr in (0.05,0.1):
        MODELS[f'hgb_d{d}_lr{lr}']=HistGradientBoostingClassifier(max_depth=d,learning_rate=lr,max_iter=150,random_state=0)
gkf=GroupKFold(n_splits=5)
folds=list(gkf.split(train,groups=train.student_key))
y=train.passed_quiz.values
oof={}; rows=[]
for fs,cols in FS.items():
    X=train[cols]
    for name,m in MODELS.items():
        p=np.zeros(len(train)); aucs=[]
        for a,b in folds:
            mm=clone(m).fit(X.iloc[a],y[a]); p[b]=mm.predict_proba(X.iloc[b])[:,1]
            aucs.append(roc_auc_score(y[b],p[b]))
        oof[(fs,name)]=p; rows.append((fs,name,np.mean(aucs),np.std(aucs)))
cv=pd.DataFrame(rows,columns=['features','model','cv_auc','sd']).sort_values('cv_auc',ascending=False); cv

## 3. Choice (cross-validation only)

In [ ]:
best=cv[cv.features=='with_session'].iloc[0]; print(best.to_dict())
p=oof[(best.features,best.model)]
ths=np.linspace(0.2,0.7,51); accs=[accuracy_score(y,p>=t) for t in ths]
thr=float(ths[int(np.argmax(accs))]); print('threshold',thr,'oof acc',max(accs),'baseline acc',max(y.mean(),1-y.mean()))
print('pre_session best:',cv[cv.features=='pre_session'].iloc[0].to_dict())

**Choice recorded before touching the test set:** the best `with_session` model above (see output), at the OOF-accuracy-maximising threshold printed above. The `pre_session` set is reported for context only.

## 4. Refit on all training rows, score the test set once

In [ ]:
final=clone(MODELS[best.model]).fit(train[FS[best.features]],y)
pt=final.predict_proba(test[FS[best.features]])[:,1]; yt=test.passed_quiz.values
print('TEST AUC',round(roc_auc_score(yt,pt),3),'| accuracy',round(accuracy_score(yt,pt>=thr),3),
      '| F1',round(f1_score(yt,pt>=thr),3),'| majority-class accuracy',round(max(yt.mean(),1-yt.mean()),3))
rng=np.random.default_rng(0); ids=test.student_key.unique(); g=test.groupby('student_key').indices
bs=[]
for _ in range(500):
    s=np.concatenate([g[i] for i in rng.choice(ids,len(ids))]); bs.append(roc_auc_score(yt[s],pt[s]))
print('student-level bootstrap 95% CI for AUC',np.percentile(bs,[2.5,97.5]).round(3))

## Conclusion
See the test numbers above (scored once). Read them against the majority-class accuracy and a chance-level AUC of 0.5. Caveats: session features are only usable if prediction happens just before the quiz; `prior_score` varies within student, which is odd for a placement test; no session dates, so generalisation to future sessions is untested.